# Phase 2: Random Forest Baseline

Reproduces the baseline RF classifier from the **Elliptic++** paper (KDD '23).

| Step | Transactions | Actors |
|---|---|---|
| **Preprocessing** | Drop NaN, MinMaxScaler on features | Drop Time step + dupes, MinMaxScaler |
| **Unknown class** | Removed (class 3) | Removed (class 3) |
| **Labels** | licit(2) -> 0, illicit(1) -> 1 | licit(2) -> 0, illicit(1) -> 1 |
| **Split** | Timestep: train < 35, test >= 35 | 70/30 (shuffle=False, rs=15) |
| **Model** | RandomForest(n_estimators=50) | RandomForest(n_estimators=50) |
| **Features** | 183 features | 56 features |

Reference: [Elliptic++ GitHub](https://github.com/git-disl/EllipticPlusPlus)

In [ ]:
import sys, os
sys.path.insert(0, '..')

import numpy as np
import pandas as pd
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import seaborn as sns
import logging

from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import (
    classification_report, confusion_matrix, f1_score,
    precision_recall_fscore_support, roc_auc_score, roc_curve,
)
from sklearn.preprocessing import MinMaxScaler

from src.models.baseline import (
    load_transaction_features, load_actor_features,
    preprocess_transactions, preprocess_actors,
    split_transactions, split_actors,
    train_rf_transactions, train_rf_actors, ModelResults,
)
from src.explain.importance import (
    get_feature_importance, get_permutation_importance,
    get_shap_values, plot_top_features, plot_shap_summary,
)

logging.basicConfig(level=logging.INFO, format='%(levelname)s: %(message)s')
RESULTS_DIR = '../docs'
os.makedirs(RESULTS_DIR, exist_ok=True)
print('Environment loaded')

## 1. Data Loading & Exploration

In [ ]:
tx_raw = load_transaction_features()
actor_raw = load_actor_features()

print('=== Raw Data ===')
print(f'Transactions: {tx_raw.shape[0]:,} rows x {tx_raw.shape[1]} columns')
print(f'  Class distribution: {tx_raw["class"].value_counts().sort_index().to_dict()}')
print(f'  NaN rows: {tx_raw.isnull().any(axis=1).sum():,}')
print()
print(f'Actors: {actor_raw.shape[0]:,} rows x {actor_raw.shape[1]} columns')
print(f'  Class distribution: {actor_raw["class"].value_counts().sort_index().to_dict()}')
print(f'  Duplicate rows: {actor_raw.duplicated().sum():,}')

## 2. Preprocessing

In [ ]:
tx = preprocess_transactions(tx_raw)
print('=== Transactions (after preprocessing) ===')
print(f'Shape: {tx.shape}')
print(f'Class distribution: {tx["class"].value_counts().to_dict()}')
print(f'Illicit ratio: {tx["class"].mean():.3%}')

act = preprocess_actors(actor_raw)
print()
print('=== Actors (after preprocessing) ===')
print(f'Shape: {act.shape}')
print(f'Class distribution: {act["class"].value_counts().to_dict()}')
print(f'Illicit ratio: {act["class"].mean():.3%}')

## 3. Train / Test Splits

In [ ]:
X_train_tx, X_test_tx, y_train_tx, y_test_tx = split_transactions(tx)
print('=== Transaction Splits ===')
print(f'Train: {len(X_train_tx):,} ({X_train_tx.shape[1]} features)')
print(f'Test:  {len(X_test_tx):,}')
print(f'Train illicit ratio: {y_train_tx.mean():.3%}')
print(f'Test illicit ratio:  {y_test_tx.mean():.3%}')

X_train_act, X_test_act, y_train_act, y_test_act = split_actors(act)
print()
print('=== Actor Splits ===')
print(f'Train: {len(X_train_act):,} ({X_train_act.shape[1]} features)')
print(f'Test:  {len(X_test_act):,}')
print(f'Train illicit ratio: {y_train_act.mean():.3%}')
print(f'Test illicit ratio:  {y_test_act.mean():.3%}')

## 4. Random Forest Training - Transactions

In [ ]:
tx_results = train_rf_transactions(n_estimators=50, random_state=42)
print(tx_results.report)

In [ ]:
cm_tx = confusion_matrix(tx_results.y_test, tx_results.y_pred)
fig, ax = plt.subplots(figsize=(6, 5))
sns.heatmap(
    cm_tx, annot=True, fmt='d', cmap='Blues',
    xticklabels=['Licit', 'Illicit'],
    yticklabels=['Licit', 'Illicit'],
    ax=ax, cbar_kws={'label': 'Count'},
)
ax.set_xlabel('Predicted')
ax.set_ylabel('Actual')
ax.set_title('RF - Transactions Confusion Matrix', fontsize=14, fontweight='bold', pad=12)
fig.tight_layout()
path = os.path.join(RESULTS_DIR, 'confusion_matrix_transactions.png')
fig.savefig(path, dpi=150, bbox_inches='tight')
plt.close(fig)
print(f'Saved: {path}')

## 5. Random Forest Training - Actors

In [ ]:
actor_results = train_rf_actors(n_estimators=50, random_state=42, model_random_state=42)
print(actor_results.report)

In [ ]:
cm_act = confusion_matrix(actor_results.y_test, actor_results.y_pred)
fig, ax = plt.subplots(figsize=(6, 5))
sns.heatmap(
    cm_act, annot=True, fmt='d', cmap='Oranges',
    xticklabels=['Licit', 'Illicit'],
    yticklabels=['Licit', 'Illicit'],
    ax=ax, cbar_kws={'label': 'Count'},
)
ax.set_xlabel('Predicted')
ax.set_ylabel('Actual')
ax.set_title('RF - Actors Confusion Matrix', fontsize=14, fontweight='bold', pad=12)
fig.tight_layout()
path = os.path.join(RESULTS_DIR, 'confusion_matrix_actors.png')
fig.savefig(path, dpi=150, bbox_inches='tight')
plt.close(fig)
print(f'Saved: {path}')

## 6. Feature Importance - Transactions

In [ ]:
tx_fi = get_feature_importance(tx_results.model, tx_results.feature_names)
print('Top 15 Transaction Features (RF Importance):')
print(tx_fi.head(15).to_string(index=False))

plot_top_features(
    tx_fi, top_n=20,
    title='Top 20 Transaction Feature Importances (RF)',
    output_path=os.path.join(RESULTS_DIR, 'feature_importance_transactions.png'),
)

In [ ]:
tx_perm = get_permutation_importance(
    tx_results.model,
    X_test_tx.values,
    y_test_tx.values,
    tx_results.feature_names,
    n_repeats=10,
)
print('Top 15 Transaction Features (Permutation Importance):')
print(tx_perm.head(15).to_string(index=False))

## 7. Feature Importance - Actors

In [ ]:
act_fi = get_feature_importance(actor_results.model, actor_results.feature_names)
print('Top 15 Actor Features (RF Importance):')
print(act_fi.head(15).to_string(index=False))

plot_top_features(
    act_fi, top_n=20,
    title='Top 20 Actor Feature Importances (RF)',
    output_path=os.path.join(RESULTS_DIR, 'feature_importance_actors.png'),
)

In [ ]:
act_perm = get_permutation_importance(
    actor_results.model,
    X_test_act.values,
    y_test_act.values,
    actor_results.feature_names,
    n_repeats=10,
)
print('Top 15 Actor Features (Permutation Importance):')
print(act_perm.head(15).to_string(index=False))

## 8. SHAP Analysis (optional)

In [ ]:
shap_explainer, shap_values, X_shap = get_shap_values(
    tx_results.model,
    X_test_tx.values,
    tx_results.feature_names,
    max_samples=500,
)

if shap_explainer is not None:
    plot_shap_summary(
        shap_values,
        X_shap,
        tx_results.feature_names,
        output_path=os.path.join(RESULTS_DIR, 'shap_summary_transactions.png'),
        max_display=20,
    )
    print('SHAP summary saved')
else:
    print('SHAP not available')

## 9. ROC Curves

In [ ]:
tx_probs = tx_results.model.predict_proba(X_test_tx.values)[:, 1]
tx_fpr, tx_tpr, _ = roc_curve(tx_results.y_test, tx_probs)
tx_auc = roc_auc_score(tx_results.y_test, tx_probs)

act_probs = actor_results.model.predict_proba(X_test_act.values)[:, 1]
act_fpr, act_tpr, _ = roc_curve(actor_results.y_test, act_probs)
act_auc = roc_auc_score(actor_results.y_test, act_probs)

fig, axes = plt.subplots(1, 2, figsize=(14, 6))

axes[0].plot(tx_fpr, tx_tpr, color='#3498db', lw=2,
             label=f'RF Transactions (AUC={tx_auc:.3f})')
axes[0].plot([0, 1], [0, 1], color='gray', lw=1, linestyle='--')
axes[0].set_xlabel('False Positive Rate')
axes[0].set_ylabel('True Positive Rate')
axes[0].set_title('ROC - Transactions')
axes[0].legend(loc='lower right')
axes[0].grid(True, alpha=0.3)

axes[1].plot(act_fpr, act_tpr, color='#e67e22', lw=2,
             label=f'RF Actors (AUC={act_auc:.3f})')
axes[1].plot([0, 1], [0, 1], color='gray', lw=1, linestyle='--')
axes[1].set_xlabel('False Positive Rate')
axes[1].set_ylabel('True Positive Rate')
axes[1].set_title('ROC - Actors')
axes[1].legend(loc='lower right')
axes[1].grid(True, alpha=0.3)

fig.suptitle('ROC Curves - Random Forest Baseline', fontsize=16, fontweight='bold', y=1.02)
fig.tight_layout()
path = os.path.join(RESULTS_DIR, 'roc_curves.png')
fig.savefig(path, dpi=150, bbox_inches='tight')
plt.close(fig)
print(f'Saved: {path}')

## 10. Paper Comparison

In [ ]:
results = {
    'Transactions (our RF)': {
        'Precision': round(tx_results.precision, 3),
        'Recall':    round(tx_results.recall, 3),
        'F1':        round(tx_results.f1, 3),
        'AUC-ROC':   round(tx_auc, 3),
    },
    'Transactions (paper RF)': {
        'Precision': 0.986,
        'Recall':    0.727,
        'F1':        0.833,
        'AUC-ROC':   0.986,
    },
    'Actors (our RF)': {
        'Precision': round(actor_results.precision, 3),
        'Recall':    round(actor_results.recall, 3),
        'F1':        round(actor_results.f1, 3),
        'AUC-ROC':   round(act_auc, 3),
    },
    'Actors (paper RF)': {
        'Precision': 0.921,
        'Recall':    0.802,
        'F1':        0.857,
        'AUC-ROC':   0.980,
    },
}
comparison = pd.DataFrame(results).T
print('Results Comparison')
print(comparison.to_string())

## 11. Summary

In [ ]:
print('=' * 60)
print('PHASE 2 COMPLETE - Random Forest Baseline')
print('=' * 60)
print()
print('Transactions RF:')
print(f'   Precision: {tx_results.precision:.3f} | Recall: {tx_results.recall:.3f} | F1: {tx_results.f1:.3f}')
print(f'   AUC-ROC: {tx_auc:.3f}')
print()
print('Actors RF:')
print(f'   Precision: {actor_results.precision:.3f} | Recall: {actor_results.recall:.3f} | F1: {actor_results.f1:.3f}')
print(f'   AUC-ROC: {act_auc:.3f}')
print()
print('Output artifacts in docs/')
for f in sorted(os.listdir(RESULTS_DIR)):
    if f.endswith('.png'):
        size = os.path.getsize(os.path.join(RESULTS_DIR, f)) / 1024
        print(f'   -> {f} ({size:.0f} KB)')
print()
print('Next: Phase 3 - GNN models (GraphSAGE, GAT)')
print('   Requires approval before proceeding.')